# AP Bio Misconception-Tagged Item Generator — full pipeline

Run top to bottom. Trains a Qwen3-1.7B (QLoRA) to generate MCQs where every
distractor is a named misconception, evaluates base-vs-tuned (in-dist + OOD +
frontier), pushes the model to HF, and launches a live demo.

**First:** Runtime > Change runtime type > **GPU**.

## 1. Setup

In [ ]:
!nvidia-smi -L  # confirm a GPU is attached

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl datasets
!pip install -q gradio

## 2. Get the code + data (v2, 5 topics)

In [ ]:
REPO = "https://github.com/gabriel-xiong/QuestionGen.git"
import os
if not os.path.isdir("QuestionGen"):
    !git clone $REPO
%cd QuestionGen
!git pull -q
print("train examples:")
!wc -l data/gen_sft_train.jsonl   # expect ~1900+ (v2, 5 topics)

## 3. Train (QLoRA, completion-only masking)

In [ ]:
!python scripts/train_gen_sft.py \n    --model Qwen/Qwen3-1.7B \n    --train data/gen_sft_train.jsonl \n    --val   data/gen_sft_val.jsonl \n    --out   outputs/gen_lora \n    --epochs 3 --batch-size 4 --grad-accum 4 --max-seq-length 1536

## 4. Save the adapter to Google Drive (survives disconnects)

In [ ]:
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
DEST='/content/drive/MyDrive/apbio_item_generator'; os.makedirs(DEST, exist_ok=True)
for src in ['outputs/gen_lora','outputs/gen_merged','data/eval_results.jsonl']:
    if not os.path.exists(src):
        continue
    d = os.path.join(DEST, os.path.basename(src))
    if os.path.isdir(src):
        shutil.copytree(src, d, dirs_exist_ok=True)
    else:
        shutil.copy2(src, d)
    print('saved ->', d)
print('Drive folder:', DEST)

## 5. Sanity check — generate one item from a held-out scenario

In [ ]:
import json, sys; sys.path.insert(0,'scripts')
from unsloth import FastLanguageModel
import gen_spec
model, tok = FastLanguageModel.from_pretrained('outputs/gen_lora', max_seq_length=1536, load_in_4bit=True)
FastLanguageModel.for_inference(model)
sc = json.loads(open('data/eval_scenarios.jsonl').readline())
system, user = gen_spec.build_generation_prompt(sc['topic'], sc['misconception_ids'])
txt = tok.apply_chat_template([{'role':'system','content':system},{'role':'user','content':user}], tokenize=False, add_generation_prompt=True)
ids = tok(txt, return_tensors='pt').to(model.device)
out = model.generate(**ids, max_new_tokens=512, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True))

## 6. Merge the adapter (so eval / HF can load it with plain transformers)

In [ ]:
model.save_pretrained_merged('outputs/gen_merged', tok, save_method='merged_16bit')

## 7. Set your OpenAI key (for the conceptual judge)
Genetics is scored offline; conceptual dimensions use the calibrated gpt-4o judge.

In [ ]:
import os, getpass
os.environ['OPENAI_API_KEY'] = getpass.getpass('Paste OpenAI key: ').strip()
assert os.environ['OPENAI_API_KEY'].startswith('sk-'), 'KEY EMPTY/WRONG'

## 8. Base vs tuned — IN-DISTRIBUTION (header must read gpt-4o, not MOCK)

In [ ]:
!python scripts/eval_generation.py \n    --base hf:Qwen/Qwen3-1.7B --tuned hf:outputs/gen_merged \n    --judge-model gpt-4o --out data/eval_results.jsonl

## 9. Base vs tuned — OUT-OF-DISTRIBUTION (photosynthesis, never trained)

In [ ]:
!python scripts/eval_generation.py \n    --base hf:Qwen/Qwen3-1.7B --tuned hf:outputs/gen_merged \n    --scenarios data/eval_scenarios_ood.jsonl --temperature 0.7 \n    --judge-model gpt-4o --out data/eval_results_ood.jsonl

## 10. Frontier comparison — prompted gpt-4o vs your tuned model
In-distribution (genetics is an objective, bias-free anchor).

In [ ]:
!python scripts/eval_generation.py \n    --base openai:gpt-4o --tuned hf:outputs/gen_merged \n    --judge-model gpt-4o --out data/eval_frontier.jsonl

## 11. Push the MERGED model to the HF Hub

In [ ]:
from huggingface_hub import login
login()  # paste your hf_... WRITE token
model.push_to_hub_merged('gabriel-xiong/apbio-item-generator-qwen3-1.7b', tok, save_method='merged_16bit')

## 12. VIDEO money shot — base vs tuned on the SAME prompt

In [ ]:
import json, sys; sys.path.insert(0,'scripts')
import gen_spec
from unsloth import FastLanguageModel
sc = json.loads(open('data/eval_scenarios_ood.jsonl').readline())
system, user = gen_spec.build_generation_prompt(sc['topic'], sc['misconception_ids'])
msgs=[{'role':'system','content':system},{'role':'user','content':user}]
def run(path):
    m,t = FastLanguageModel.from_pretrained(path, max_seq_length=1536, load_in_4bit=True)
    FastLanguageModel.for_inference(m)
    x=t.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    i=t(x, return_tensors='pt').to(m.device)
    o=m.generate(**i, max_new_tokens=512, do_sample=False, pad_token_id=t.eos_token_id)
    return t.decode(o[0][i['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== BASE Qwen3-1.7B ===
', run('Qwen/Qwen3-1.7B'))
print('
=== TUNED ===
', run('outputs/gen_lora'))

## 13. Live interactive demo (public share link)
Prints a https://…gradio.live URL. Uses your merged HF model (set in scripts/app.py).

In [ ]:
!sed -i 's/demo.launch()/demo.launch(share=True)/' scripts/app.py
!python scripts/app.py

## Submission checklist
- [ ] Dataset — published (this repo, `data/`)
- [ ] Model — on HF Hub (step 11)
- [ ] Running demo — HF Space (see `docs/demo.md`) or the share link (step 13)
- [ ] Eval results — steps 8–10; tables in `docs/brainlift_generator.md`
- [ ] Brainlift — `docs/brainlift_generator.md`
- [ ] 3–5 min demo video